In [ ]:
import os
from pathlib import Path

root = Path("/kaggle/input")
split_names = {"train", "val", "valid", "validation", "test"}

for folder, subfolders, files in os.walk(root):
    folder = Path(folder)

    if "img_text" in subfolders:
        print("Image and text folder:", folder / "img_text")
        subfolders.remove("img_text")

    for name in sorted(files):
        if Path(name).stem.lower() in split_names:
            print("Split file:", folder / name)

In [ ]:
import re
from itertools import islice
from pathlib import Path

base = Path("/kaggle/input/datasets/vanshika205")
annotation_path = base / "text-data/data/enem/train.txt"
media_dir = base / "image-text/img_text"

# Read just the first part of the annotation file.
with annotation_path.open(encoding="utf-8-sig") as file:
    preview = file.read(1500)

print("Annotation preview:")
print(preview[:700])

match = re.search(r"(?m)^\s*(\d+)\b", preview)

if match:
    sample_id = match.group(1)
    print("\nSample ID:", sample_id)

    text_path = media_dir / f"{sample_id}.txt"
    image_paths = [media_dir / f"{sample_id}{ext}" for ext in (".jpg", ".jpeg", ".png")]
    image_path = next((path for path in image_paths if path.is_file()), None)

    print("Image:", image_path or "No direct match in img_text")
    print("Text:", text_path if text_path.is_file() else "No direct match in img_text")

    if not image_path or not text_path.is_file():
        print("\nFirst few entries inside img_text:")
        for path in islice(media_dir.iterdir(), 10):
            print(path.name)
else:
    print("\nCould not identify a sample ID from the preview.")
    

In [ ]:
from ast import literal_eval

from IPython.display import display
from PIL import Image

with annotation_path.open(encoding="utf-8-sig") as file:
    first_row = next(line for line in file if line.strip())

sample_id, raw_labels = first_row.strip().split("\t", 1)
annotations = literal_eval(raw_labels)

text = (media_dir / f"{sample_id}.txt").read_text(encoding="utf-8-sig")
print(f"Original post for {sample_id}:")
print(text.strip())

print("\nAnnotated entities and emotions:")
for entity, emotions in annotations:
    print(f"{entity}: {', '.join(emotions)}")

with Image.open(media_dir / f"{sample_id}.jpg") as image:
    image.thumbnail((450, 450))
    display(image)

In [ ]:
# Read the training annotations and count posts, entity annotations, and emotion labels.
# Count each emotion once per entity annotation and print the 30 most common labels.

from ast import literal_eval
from collections import Counter

emotion_counts = Counter()
post_count = 0
entity_count = 0

with annotation_path.open(encoding="utf-8-sig") as file:
    for line in file:
        if not line.strip():
            continue

        _, raw_labels = line.strip().split("\t", 1)
        annotations = literal_eval(raw_labels)
        post_count += 1

        for entity, emotions in annotations:
            entity_count += 1
            emotion_counts.update({
                emotion.strip().lower()
                for emotion in emotions
                if emotion.strip()
            })

print("Training posts:", post_count)
print("Entity annotations:", entity_count)
print("Unique emotion labels:", len(emotion_counts))
print("\n30 most common labels:")

for emotion, count in emotion_counts.most_common(30):
    print(f"{emotion}: {count}")

In [ ]:
# Check how many training entity annotations have one explicit sentiment label,
# conflicting sentiment labels, or no explicit sentiment label.

from ast import literal_eval
from collections import Counter

sentiments = {"positive", "negative", "neutral"}
coverage = Counter()

with annotation_path.open(encoding="utf-8-sig") as file:
    for line in file:
        if not line.strip():
            continue

        _, raw_labels = line.strip().split("\t", 1)

        for entity, emotions in literal_eval(raw_labels):
            labels = {emotion.strip().lower() for emotion in emotions}
            explicit = labels & sentiments

            if len(explicit) == 1:
                coverage[f"Only {next(iter(explicit))}"] += 1
            elif explicit:
                coverage["Conflicting explicit labels"] += 1
            else:
                coverage["No explicit sentiment label"] += 1

total = sum(coverage.values())

for group in ("Only positive", "Only negative", "Only neutral",
              "Conflicting explicit labels", "No explicit sentiment label"):
    count = coverage[group]
    print(f"{group}: {count:,} ({count / total:.1%})")
    

In [ ]:
# Estimate training-label coverage with a small draft emotion-to-sentiment mapping.
# Count conflicting and unmapped annotations separately for review.

from ast import literal_eval
from collections import Counter

emotion_groups = {
    "positive": {
        "positive", "positivity", "admiration", "respect", "affection",
        "gratitude", "joy", "happiness", "hope", "optimism", "optimistic",
    },
    "negative": {
        "negative", "sadness", "disappointment", "disillusionment",
        "anger", "anxiety", "worry", "disapproval",
    },
    "neutral": {"neutral"},
}

coverage = Counter()
unmapped_emotions = Counter()

with annotation_path.open(encoding="utf-8-sig") as file:
    for line in file:
        if not line.strip():
            continue

        _, raw_labels = line.rstrip("\r\n").split("\t", 1)
        for _, emotions in literal_eval(raw_labels):
            labels = {emotion.strip().lower() for emotion in emotions if emotion.strip()}
            matched = {
                sentiment
                for sentiment, group in emotion_groups.items()
                if labels & group
            }

            if len(matched) == 1:
                coverage[next(iter(matched))] += 1
            elif matched:
                coverage["conflicting"] += 1
            else:
                coverage["unmapped"] += 1
                unmapped_emotions.update(labels)

total = sum(coverage.values())

for group in ("positive", "negative", "neutral", "conflicting", "unmapped"):
    count = coverage[group]
    print(f"{group}: {count:,} ({count / total:.1%})")

print("\nMost common emotions in unmapped annotations:")
for emotion, count in unmapped_emotions.most_common(10):
    print(f"{emotion}: {count:,}")

In [ ]:
# Build a draft training table with one row per entity annotation.
# Keep rows that map to exactly one sentiment and retain the original emotions.

from ast import literal_eval
import pandas as pd

rows = []

with annotation_path.open(encoding="utf-8-sig") as file:
    for line in file:
        if not line.strip():
            continue

        sample_id, raw_labels = line.rstrip("\r\n").split("\t", 1)

        for entity, emotions in literal_eval(raw_labels):
            labels = {emotion.strip().lower() for emotion in emotions}
            matched = [
                sentiment
                for sentiment, group in emotion_groups.items()
                if labels & group
            ]

            if len(matched) == 1:
                rows.append({
                    "sample_id": sample_id,
                    "entity": entity,
                    "sentiment": matched[0],
                    "raw_emotions": emotions,
                })

train_draft = pd.DataFrame(rows)

print(f"Rows: {len(train_draft):,}")
print(f"Unique posts: {train_draft['sample_id'].nunique():,}")
print(train_draft["sentiment"].value_counts())
display(train_draft.head())

In [ ]:
# Index image and text files by post ID, then attach their paths to the draft table.
# Report posts with missing files and keep complete rows in a new table.

import os
from pathlib import Path

post_ids = set(train_draft["sample_id"])
text_paths = {}
image_paths = {}
image_priority = {".jpg": 0, ".jpeg": 1, ".png": 2}

with os.scandir(media_dir) as entries:
    for entry in entries:
        path = Path(entry.name)
        post_id = path.stem
        ext = path.suffix.lower()

        if post_id not in post_ids or not entry.is_file():
            continue

        if ext == ".txt":
            text_paths[post_id] = entry.path
        elif ext in image_priority:
            current = image_paths.get(post_id)
            if current is None or image_priority[ext] < image_priority[Path(current).suffix.lower()]:
                image_paths[post_id] = entry.path

train_with_paths = train_draft.assign(
    text_path=train_draft["sample_id"].map(text_paths),
    image_path=train_draft["sample_id"].map(image_paths),
)

complete = train_with_paths[["text_path", "image_path"]].notna().all(axis=1)
train_ready = train_with_paths.loc[complete].copy()

print(f"Posts missing text: {len(post_ids - text_paths.keys()):,}")
print(f"Posts missing an image: {len(post_ids - image_paths.keys()):,}")
print(f"Rows missing either file: {(~complete).sum():,}")
print(f"Rows with both files: {len(train_ready):,} / {len(train_draft):,}")


In [ ]:
# Build validation and test tables with the same label rules as training.
# Check the original splits for shared post IDs and report missing files.

from ast import literal_eval
from itertools import combinations

split_ids = {}
split_tables = {"train": train_ready}

for split in ("train", "val", "test"):
    ids = set()
    rows = []
    split_path = annotation_path.with_name(f"{split}.txt")

    with split_path.open(encoding="utf-8-sig") as file:
        for line in file:
            if not line.strip():
                continue

            sample_id, raw_labels = line.rstrip("\r\n").split("\t", 1)
            ids.add(sample_id)

            if split == "train":
                continue  # Reuse the training table we already built.

            for entity, emotions in literal_eval(raw_labels):
                labels = {emotion.strip().lower() for emotion in emotions}
                matched = [
                    sentiment for sentiment, group in emotion_groups.items()
                    if labels & group
                ]
                if len(matched) == 1:
                    rows.append((sample_id, entity, matched[0], emotions))

    split_ids[split] = ids

    if split != "train":
        split_tables[split] = pd.DataFrame(
            rows, columns=["sample_id", "entity", "sentiment", "raw_emotions"]
        )

overlap_found = False
for left, right in combinations(split_ids, 2):
    overlap = split_ids[left] & split_ids[right]
    print(f"{left} / {right} shared posts: {len(overlap):,}")
    overlap_found |= bool(overlap)

assert not overlap_found, "Resolve split overlap before training."

for split in ("val", "test"):
    table = split_tables[split]
    post_files = {}

    for sample_id in table["sample_id"].unique():
        text_path = media_dir / f"{sample_id}.txt"
        image_path = next(
            (path for ext in (".jpg", ".jpeg", ".png")
             if (path := media_dir / f"{sample_id}{ext}").is_file()),
            None,
        )
        if text_path.is_file() and image_path is not None:
            post_files[sample_id] = (str(text_path), str(image_path))

    ready = table.loc[table["sample_id"].isin(post_files)].copy()
    ready["text_path"] = ready["sample_id"].map({key: pair[0] for key, pair in post_files.items()})
    ready["image_path"] = ready["sample_id"].map({key: pair[1] for key, pair in post_files.items()})
    split_tables[split] = ready

    print(f"\n{split}: {len(ready):,} rows ready, {len(table) - len(ready):,} missing files")
    print(ready["sentiment"].value_counts())

In [ ]:
# Set the shared hyperparameters for all three fusion models.
# Build the data loaders and check the shapes of one training batch.

from pathlib import Path

import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision.models import MobileNet_V3_Small_Weights
from transformers import AutoTokenizer, set_seed

# Data settings
TEXT_MODEL = "distilbert/distilbert-base-uncased"
LABEL_TO_ID = {"negative": 0, "neutral": 1, "positive": 2}
MAX_TEXT_LENGTH = 128
BATCH_SIZE = 16
SEED = 42

# Settings used later in the training loop
MAX_EPOCHS = 10
GRAD_ACCUM_STEPS = 2
ENCODER_LR = 2e-5
FUSION_HEAD_LR = 1e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
EARLY_STOPPING_PATIENCE = 3
MAX_GRAD_NORM = 1.0

set_seed(SEED)

tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL)
image_transform = MobileNet_V3_Small_Weights.DEFAULT.transforms()


class SentimentDataset(Dataset):
    def __init__(self, table):
        self.table = table.reset_index(drop=True)

    def __len__(self):
        return len(self.table)

    def __getitem__(self, index):
        row = self.table.iloc[index]
        text = Path(row["text_path"]).read_text(encoding="utf-8-sig").strip()

        with Image.open(row["image_path"]) as image:
            pixels = image_transform(image.convert("RGB"))

        return row["entity"], text, pixels, LABEL_TO_ID[row["sentiment"]]


def collate_batch(examples):
    entities, texts, images, labels = zip(*examples)

    batch = tokenizer(
        list(entities),
        list(texts),
        padding=True,
        truncation=True,
        max_length=MAX_TEXT_LENGTH,
        return_token_type_ids=False,
        return_tensors="pt",
    )

    batch["pixel_values"] = torch.stack(images)
    batch["labels"] = torch.tensor(labels, dtype=torch.long)
    return dict(batch)


loaders = {
    split: DataLoader(
        SentimentDataset(table),
        batch_size=BATCH_SIZE,
        shuffle=(split == "train"),
        collate_fn=collate_batch,
        num_workers=0,
    )
    for split, table in split_tables.items()
}

batch = next(iter(loaders["train"]))

for name, tensor in batch.items():
    print(f"{name}: {tuple(tensor.shape)}")

print("Label IDs:", batch["labels"].tolist())

In [ ]:
# Build Early Fusion by concatenating DistilBERT and MobileNetV3-Small features.
# Run one forward and backward pass to check the output shape, loss, and gradients.

from torch import nn
from torchvision.models import mobilenet_v3_small
from transformers import AutoModel


class EarlyFusion(nn.Module):
    def __init__(self):
        super().__init__()

        self.text_encoder = AutoModel.from_pretrained(TEXT_MODEL)

        mobilenet = mobilenet_v3_small(
            weights=MobileNet_V3_Small_Weights.DEFAULT
        )
        self.image_encoder = mobilenet.features
        self.image_pool = nn.AdaptiveAvgPool2d(1)

        text_dim = self.text_encoder.config.hidden_size
        image_dim = mobilenet.classifier[0].in_features
        self.head = nn.Linear(text_dim + image_dim, len(LABEL_TO_ID))

    def forward(self, input_ids, attention_mask, pixel_values):
        text_output = self.text_encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
        text_features = text_output.last_hidden_state[:, 0]

        image_features = self.image_encoder(pixel_values)
        image_features = self.image_pool(image_features).flatten(1)

        fused_features = torch.cat((text_features, image_features), dim=1)
        return self.head(fused_features)


assert torch.cuda.is_available(), "Enable a Kaggle GPU before running this cell."
device = torch.device("cuda")

set_seed(SEED)
model = EarlyFusion().to(device)
model.train()

gpu_batch = {name: tensor.to(device) for name, tensor in batch.items()}
labels = gpu_batch.pop("labels")

logits = model(**gpu_batch)
loss = nn.CrossEntropyLoss()(logits, labels)

assert logits.shape == (labels.size(0), len(LABEL_TO_ID))
assert torch.isfinite(loss).item(), "The loss is not finite."

loss.backward()

print("Logits shape:", tuple(logits.shape))
print("Loss:", round(loss.item(), 4))
print("Text encoder received gradients:", next(model.text_encoder.parameters()).grad is not None)
print("Image encoder received gradients:", next(model.image_encoder.parameters()).grad is not None)
print("Prediction head received gradients:", model.head.weight.grad is not None)

model.zero_grad(set_to_none=True)

In [ ]:
# Train Early Fusion with the shared settings and validate after each epoch.
# Save the best weights by validation macro F1 and stop after three stale epochs.

from math import ceil
from pathlib import Path

from sklearn.metrics import f1_score
from transformers import get_linear_schedule_with_warmup

set_seed(SEED)
model = EarlyFusion().to(device)

optimizer = torch.optim.AdamW(
    [
        {"params": model.text_encoder.parameters(), "lr": ENCODER_LR},
        {"params": model.image_encoder.parameters(), "lr": ENCODER_LR},
        {"params": model.head.parameters(), "lr": FUSION_HEAD_LR},
    ],
    weight_decay=WEIGHT_DECAY,
)

train_loader = loaders["train"]
val_loader = loaders["val"]
num_batches = len(train_loader)
num_examples = len(train_loader.dataset)

updates_per_epoch = ceil(num_batches / GRAD_ACCUM_STEPS)
total_updates = MAX_EPOCHS * updates_per_epoch
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(WARMUP_RATIO * total_updates),
    num_training_steps=total_updates,
)

use_fp16 = device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_fp16)
loss_fn = nn.CrossEntropyLoss(reduction="sum")

checkpoint_path = Path("/kaggle/working/checkpoints/early_best.pt")
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)

best_val_f1 = float("-inf")
best_epoch = 0
epochs_without_improvement = 0

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    train_loss = torch.zeros((), device=device)

    for step, batch_data in enumerate(train_loader):
        inputs = {name: tensor.to(device) for name, tensor in batch_data.items()}
        targets = inputs.pop("labels")

        # Normalize by the actual group size, including the final partial batch.
        group_start = (step // GRAD_ACCUM_STEPS) * GRAD_ACCUM_STEPS
        group_examples = min(
            BATCH_SIZE * GRAD_ACCUM_STEPS,
            num_examples - group_start * BATCH_SIZE,
        )

        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_fp16):
            loss = loss_fn(model(**inputs), targets)

        train_loss += loss.detach()
        scaler.scale(loss / group_examples).backward()

        end_of_group = (step + 1) % GRAD_ACCUM_STEPS == 0
        if end_of_group or step + 1 == num_batches:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            previous_scale = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()

            # Advance the scheduler only when the optimizer updates the weights.
            if scaler.get_scale() >= previous_scale:
                scheduler.step()

            optimizer.zero_grad(set_to_none=True)

    model.eval()
    actual, predicted = [], []

    with torch.inference_mode():
        for batch_data in val_loader:
            actual.extend(batch_data["labels"].tolist())
            inputs = {
                name: tensor.to(device)
                for name, tensor in batch_data.items()
                if name != "labels"
            }

            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_fp16):
                logits = model(**inputs)

            predicted.extend(logits.argmax(dim=1).cpu().tolist())

    val_f1 = f1_score(
        actual, predicted, labels=list(LABEL_TO_ID.values()),
        average="macro", zero_division=0,
    )
    print(
        f"Epoch {epoch}/{MAX_EPOCHS} | "
        f"train loss: {train_loss.item() / num_examples:.4f} | "
        f"val macro F1: {val_f1:.4f}",
        flush=True,
    )

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_epoch = epoch
        epochs_without_improvement = 0
        torch.save(model.state_dict(), checkpoint_path)
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            print("Early stopping.")
            break

print(f"Best validation macro F1: {best_val_f1:.4f} at epoch {best_epoch}")
print(f"Checkpoint: {checkpoint_path}")

In [ ]:
# Reload the best Early Fusion checkpoint and evaluate it on the held-out test split.
# Report accuracy, macro F1, and precision and recall for each sentiment.

from sklearn.metrics import accuracy_score, classification_report, f1_score

assert checkpoint_path.is_file(), f"Checkpoint missing: {checkpoint_path}"
model.load_state_dict(torch.load(checkpoint_path, map_location="cpu", weights_only=True))
model.eval()

actual, predicted = [], []

with torch.inference_mode():
    for batch_data in loaders["test"]:
        actual.extend(batch_data["labels"].tolist())
        inputs = {
            name: tensor.to(device)
            for name, tensor in batch_data.items()
            if name != "labels"
        }

        with torch.autocast(
            device_type=device.type, dtype=torch.float16, enabled=use_fp16
        ):
            logits = model(**inputs)

        predicted.extend(logits.argmax(dim=1).cpu().tolist())

label_names = sorted(LABEL_TO_ID, key=LABEL_TO_ID.get)
label_ids = [LABEL_TO_ID[name] for name in label_names]

print(f"Test accuracy: {accuracy_score(actual, predicted):.4f}")
print(f"Test macro F1: {f1_score(actual, predicted, labels=label_ids, average='macro', zero_division=0):.4f}")
print(classification_report(
    actual, predicted, labels=label_ids, target_names=label_names,
    digits=4, zero_division=0,
))

In [ ]:
# Copy the best checkpoint to the top-level output folder and link to it.
%cd /kaggle/working

from shutil import copy2
from IPython.display import FileLink, display

copy2("checkpoints/early_best.pt", "early_best.pt")
display(FileLink("early_best.pt"))

In [ ]:
# Evaluate the saved Early Fusion checkpoint on the held-out test split.
# Save four curves, two confusion matrices, and annotated prediction examples.

from pathlib import Path
from shutil import make_archive
from textwrap import fill

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import FileLink, display
from PIL import Image
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
)
from torch.utils.data import DataLoader

working_dir = Path("/kaggle/working")
input_dir = Path("/kaggle/input/datasets/vanshika205/model-1-checkpoint")
candidates = [
    working_dir / "checkpoints/early_best.pt",
    working_dir / "early_best.pt",
    *sorted(input_dir.rglob("early_best.pt")),
]
checkpoint = next((path for path in candidates if path.is_file()), None)

if checkpoint is None:
    raise FileNotFoundError(
        "Attach your Model 1 checkpoint as a Kaggle input, "
        "then add its full file path to candidates."
    )

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
early_eval_model = EarlyFusion()
early_eval_model.load_state_dict(
    torch.load(checkpoint, map_location="cpu", weights_only=True)
)
early_eval_model = early_eval_model.to(device).eval()
print(f"Loaded checkpoint: {checkpoint}", flush=True)

# Preserve test row order so each prediction matches the correct image.
test_dataset = loaders["test"].dataset
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_batch,
    num_workers=0,
)

label_names = sorted(LABEL_TO_ID, key=LABEL_TO_ID.get)
label_ids = np.array([LABEL_TO_ID[name] for name in label_names])
id_to_label = dict(zip(label_ids, label_names))
true_batches, probability_batches = [], []

with torch.inference_mode():
    for batch_data in test_loader:
        true_batches.append(batch_data["labels"].cpu().numpy())
        inputs = {
            name: tensor.to(device)
            for name, tensor in batch_data.items()
            if name != "labels"
        }

        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=device.type == "cuda",
        ):
            logits = early_eval_model(**inputs)

        probability_batches.append(logits.float().softmax(dim=1).cpu().numpy())

y_true = np.concatenate(true_batches)
probabilities = np.concatenate(probability_batches)
y_pred = label_ids[probabilities.argmax(axis=1)]
confidence = probabilities.max(axis=1)

macro_f1 = f1_score(
    y_true, y_pred, labels=label_ids, average="macro", zero_division=0
)
print(f"Test examples: {len(y_true):,}")
print(f"Test accuracy: {accuracy_score(y_true, y_pred):.4f}")
print(f"Test macro F1: {macro_f1:.4f}")
print(classification_report(
    y_true, y_pred, labels=label_ids, target_names=label_names,
    digits=4, zero_division=0,
))

output_dir = working_dir / "model1_outputs"
output_dir.mkdir(parents=True, exist_ok=True)


def save_figure(fig, filename):
    fig.tight_layout()
    fig.savefig(output_dir / filename, dpi=200, bbox_inches="tight")
    display(fig)
    plt.close(fig)


# 1–3. F1, precision, and recall against minimum prediction confidence.
# Predictions below the threshold count as misses for their true class.
thresholds = np.linspace(0, 1, 101)
above_threshold = confidence[None, :] >= thresholds[:, None]
curves = {"F1": [], "Precision": [], "Recall": []}

for label_id in label_ids:
    true_class = y_true == label_id
    kept = above_threshold & (y_pred == label_id)
    true_positives = (kept & true_class).sum(axis=1)
    prediction_count = kept.sum(axis=1)
    support = true_class.sum()

    curves["F1"].append(
        2 * true_positives / np.maximum(prediction_count + support, 1)
    )
    curves["Precision"].append(true_positives / np.maximum(prediction_count, 1))
    curves["Recall"].append(true_positives / max(support, 1))

for metric, class_curves in curves.items():
    fig, ax = plt.subplots(figsize=(7, 5))
    for label, values in zip(label_names, class_curves):
        ax.plot(thresholds, values, label=label)
    ax.plot(thresholds, np.mean(class_curves, axis=0), "k--", label="Macro average")
    ax.set(
        title=f"Model 1 | {metric} vs confidence",
        xlabel="Minimum prediction confidence",
        ylabel=metric,
        xlim=(0, 1),
        ylim=(0, 1.02),
    )
    ax.grid(alpha=0.25)
    ax.legend()
    save_figure(fig, f"{metric.lower()}_confidence.png")

# 4. One-vs-rest precision–recall curves using each class's probability.
fig, ax = plt.subplots(figsize=(7, 5))
for column, (label_id, label) in enumerate(zip(label_ids, label_names)):
    true_class = y_true == label_id
    class_scores = probabilities[:, column]
    precision, recall, _ = precision_recall_curve(true_class, class_scores)
    ap = average_precision_score(true_class, class_scores)
    ax.step(recall, precision, where="post", label=f"{label} (AP {ap:.3f})")

ax.set(
    title="Model 1 | Precision–recall",
    xlabel="Recall",
    ylabel="Precision",
    xlim=(0, 1),
    ylim=(0, 1.02),
)
ax.grid(alpha=0.25)
ax.legend()
save_figure(fig, "precision_recall.png")

# 5–6. Confusion matrices with counts and percentages within each true class.
for normalization, filename, title, fmt in [
    (None, "confusion_matrix_original.png", "Counts", "d"),
    ("true", "confusion_matrix_normalized.png", "Normalized by true class", ".1%"),
]:
    matrix = confusion_matrix(
        y_true, y_pred, labels=label_ids, normalize=normalization
    )
    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay(matrix, display_labels=label_names).plot(
        ax=ax, cmap="Blues", values_format=fmt, colorbar=False
    )
    ax.set_title(f"Model 1 | Confusion matrix ({title})")
    save_figure(fig, filename)

# 7. Show the first correct and incorrect prediction for each true class.
fig, axes = plt.subplots(len(label_ids), 2, figsize=(10, 12))

for row, label_id in enumerate(label_ids):
    for column, correct in enumerate((True, False)):
        ax = axes[row, column]
        ax.axis("off")
        matches = np.flatnonzero(
            (y_true == label_id) & ((y_pred == y_true) == correct)
        )

        if matches.size == 0:
            outcome = "correct" if correct else "incorrect"
            ax.set_title(f"No {outcome} {id_to_label[label_id]} examples")
            continue

        index = int(matches[0])
        example = test_dataset.table.iloc[index]
        with Image.open(example["image_path"]) as image:
            ax.imshow(image.convert("RGB"))

        ax.set_title(
            f"{fill(str(example['entity']), width=42)}\n"
            f"True: {id_to_label[y_true[index]]} | "
            f"Predicted: {id_to_label[y_pred[index]]} ({confidence[index]:.1%})",
            color="darkgreen" if correct else "firebrick",
        )

fig.suptitle("Model 1 | Example test predictions")
save_figure(fig, "example_predictions.png")

archive = make_archive(str(output_dir), "zip", root_dir=output_dir)
print(f"Saved all seven visuals to {archive}")

In [ ]:
%cd /kaggle/working
from IPython.display import FileLink, display

display(FileLink("model1_outputs.zip"))

In [ ]:
from pathlib import Path

checkpoint_dir = Path("/kaggle/input/datasets/vanshika205/model-1-checkpoint")
early_checkpoint = next(checkpoint_dir.rglob("early_best.pt"), None)

assert early_checkpoint is not None, "early_best.pt was not found in the uploaded dataset."
print("Model 1 checkpoint:", early_checkpoint)

In [ ]:
# Define Model 2 so text tokens can attend to spatial image features.
# Pool the fused text tokens and predict one of the three sentiment classes.

from itertools import chain

import torch
from torch import nn
from torchvision.models import MobileNet_V3_Small_Weights, mobilenet_v3_small
from transformers import AutoModel, set_seed


class CrossAttentionFusion(nn.Module):
    def __init__(self):
        super().__init__()

        self.text_encoder = AutoModel.from_pretrained(TEXT_MODEL)

        mobilenet = mobilenet_v3_small(
            weights=MobileNet_V3_Small_Weights.DEFAULT
        )
        self.image_encoder = mobilenet.features

        attention_dim = 256
        self.text_projection = nn.Linear(
            self.text_encoder.config.hidden_size, attention_dim
        )
        self.image_projection = nn.Linear(
            mobilenet.classifier[0].in_features, attention_dim
        )
        self.cross_attention = nn.MultiheadAttention(
            embed_dim=attention_dim,
            num_heads=4,
            batch_first=True,
        )
        self.fusion_norm = nn.LayerNorm(attention_dim)
        self.head = nn.Linear(attention_dim, len(LABEL_TO_ID))

    def forward(self, input_ids, attention_mask, pixel_values):
        text_tokens = self.text_encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
        ).last_hidden_state

        # Keep the spatial regions as separate tokens for attention.
        image_regions = self.image_encoder(pixel_values).flatten(2).transpose(1, 2)

        queries = self.text_projection(text_tokens)
        visual_tokens = self.image_projection(image_regions)
        attended, _ = self.cross_attention(
            queries, visual_tokens, visual_tokens, need_weights=False
        )

        fused_tokens = self.fusion_norm(queries + attended)
        mask = attention_mask.unsqueeze(-1).to(fused_tokens.dtype)
        pooled = (fused_tokens * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1)

        return self.head(pooled)

    def fusion_head_parameters(self):
        return chain(
            self.text_projection.parameters(),
            self.image_projection.parameters(),
            self.cross_attention.parameters(),
            self.fusion_norm.parameters(),
            self.head.parameters(),
        )

In [ ]:
# Fine-tune both encoders and train the attention layers using the shared settings.
# Validate each epoch and save the checkpoint with the highest validation macro F1.

from math import ceil
from pathlib import Path

from sklearn.metrics import f1_score
from transformers import get_linear_schedule_with_warmup

assert torch.cuda.is_available(), "Enable a Kaggle GPU before training."
device = torch.device("cuda")
use_fp16 = device.type == "cuda"

set_seed(SEED)
cross_model = CrossAttentionFusion().to(device)

optimizer = torch.optim.AdamW(
    [
        {"params": cross_model.text_encoder.parameters(), "lr": ENCODER_LR},
        {"params": cross_model.image_encoder.parameters(), "lr": ENCODER_LR},
        {"params": cross_model.fusion_head_parameters(), "lr": FUSION_HEAD_LR},
    ],
    weight_decay=WEIGHT_DECAY,
)

train_loader = loaders["train"]
val_loader = loaders["val"]
num_batches = len(train_loader)
num_examples = len(train_loader.dataset)

updates_per_epoch = ceil(num_batches / GRAD_ACCUM_STEPS)
total_updates = MAX_EPOCHS * updates_per_epoch

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(WARMUP_RATIO * total_updates),
    num_training_steps=total_updates,
)

scaler = torch.amp.GradScaler("cuda", enabled=use_fp16)
loss_fn = nn.CrossEntropyLoss(reduction="sum")

cross_checkpoint = Path("/kaggle/working/checkpoints/cross_attention_best.pt")
cross_checkpoint.parent.mkdir(parents=True, exist_ok=True)

best_val_f1 = float("-inf")
best_epoch = 0
epochs_without_improvement = 0


def collect_predictions(model, loader):
    model.eval()
    actual, predicted = [], []

    with torch.inference_mode():
        for batch_data in loader:
            actual.extend(batch_data["labels"].tolist())
            inputs = {
                name: tensor.to(device)
                for name, tensor in batch_data.items()
                if name != "labels"
            }

            with torch.autocast("cuda", dtype=torch.float16, enabled=use_fp16):
                logits = model(**inputs)

            predicted.extend(logits.argmax(dim=1).cpu().tolist())

    return actual, predicted


for epoch in range(1, MAX_EPOCHS + 1):
    cross_model.train()
    optimizer.zero_grad(set_to_none=True)
    train_loss = torch.zeros((), device=device)

    for step, batch_data in enumerate(train_loader):
        inputs = {name: tensor.to(device) for name, tensor in batch_data.items()}
        targets = inputs.pop("labels")

        # Normalize by the number of examples in this accumulation group.
        group_start = (step // GRAD_ACCUM_STEPS) * GRAD_ACCUM_STEPS
        group_examples = min(
            BATCH_SIZE * GRAD_ACCUM_STEPS,
            num_examples - group_start * BATCH_SIZE,
        )

        with torch.autocast("cuda", dtype=torch.float16, enabled=use_fp16):
            loss = loss_fn(cross_model(**inputs), targets)

        train_loss += loss.detach()
        scaler.scale(loss / group_examples).backward()

        end_of_group = (step + 1) % GRAD_ACCUM_STEPS == 0
        if end_of_group or step + 1 == num_batches:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(cross_model.parameters(), MAX_GRAD_NORM)

            previous_scale = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()

            if scaler.get_scale() >= previous_scale:
                scheduler.step()

            optimizer.zero_grad(set_to_none=True)

    actual, predicted = collect_predictions(cross_model, val_loader)
    val_f1 = f1_score(
        actual,
        predicted,
        labels=list(LABEL_TO_ID.values()),
        average="macro",
        zero_division=0,
    )

    print(
        f"Epoch {epoch}/{MAX_EPOCHS} | "
        f"train loss: {train_loss.item() / num_examples:.4f} | "
        f"val macro F1: {val_f1:.4f}",
        flush=True,
    )

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_epoch = epoch
        epochs_without_improvement = 0
        torch.save(cross_model.state_dict(), cross_checkpoint)
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            print("Early stopping.")
            break

print(f"Best validation macro F1: {best_val_f1:.4f} at epoch {best_epoch}")
print(f"Checkpoint: {cross_checkpoint}")


# Fine-tune both encoders and train the attention layers using the shared settings.
# Validate each epoch and save the checkpoint with the highest validation macro F1.

from math import ceil
from pathlib import Path

from sklearn.metrics import f1_score
from transformers import get_linear_schedule_with_warmup

assert torch.cuda.is_available(), "Enable a Kaggle GPU before training."
device = torch.device("cuda")
use_fp16 = device.type == "cuda"

set_seed(SEED)
cross_model = CrossAttentionFusion().to(device)

optimizer = torch.optim.AdamW(
    [
        {"params": cross_model.text_encoder.parameters(), "lr": ENCODER_LR},
        {"params": cross_model.image_encoder.parameters(), "lr": ENCODER_LR},
        {"params": cross_model.fusion_head_parameters(), "lr": FUSION_HEAD_LR},
    ],
    weight_decay=WEIGHT_DECAY,
)

train_loader = loaders["train"]
val_loader = loaders["val"]
num_batches = len(train_loader)
num_examples = len(train_loader.dataset)

updates_per_epoch = ceil(num_batches / GRAD_ACCUM_STEPS)
total_updates = MAX_EPOCHS * updates_per_epoch

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(WARMUP_RATIO * total_updates),
    num_training_steps=total_updates,
)

scaler = torch.amp.GradScaler("cuda", enabled=use_fp16)
loss_fn = nn.CrossEntropyLoss(reduction="sum")

cross_checkpoint = Path("/kaggle/working/checkpoints/cross_attention_best.pt")
cross_checkpoint.parent.mkdir(parents=True, exist_ok=True)

best_val_f1 = float("-inf")
best_epoch = 0
epochs_without_improvement = 0


def collect_predictions(model, loader):
    model.eval()
    actual, predicted = [], []

    with torch.inference_mode():
        for batch_data in loader:
            actual.extend(batch_data["labels"].tolist())
            inputs = {
                name: tensor.to(device)
                for name, tensor in batch_data.items()
                if name != "labels"
            }

            with torch.autocast("cuda", dtype=torch.float16, enabled=use_fp16):
                logits = model(**inputs)

            predicted.extend(logits.argmax(dim=1).cpu().tolist())

    return actual, predicted


for epoch in range(1, MAX_EPOCHS + 1):
    cross_model.train()
    optimizer.zero_grad(set_to_none=True)
    train_loss = torch.zeros((), device=device)

    for step, batch_data in enumerate(train_loader):
        inputs = {name: tensor.to(device) for name, tensor in batch_data.items()}
        targets = inputs.pop("labels")

        # Normalize by the number of examples in this accumulation group.
        group_start = (step // GRAD_ACCUM_STEPS) * GRAD_ACCUM_STEPS
        group_examples = min(
            BATCH_SIZE * GRAD_ACCUM_STEPS,
            num_examples - group_start * BATCH_SIZE,
        )

        with torch.autocast("cuda", dtype=torch.float16, enabled=use_fp16):
            loss = loss_fn(cross_model(**inputs), targets)

        train_loss += loss.detach()
        scaler.scale(loss / group_examples).backward()

        end_of_group = (step + 1) % GRAD_ACCUM_STEPS == 0
        if end_of_group or step + 1 == num_batches:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(cross_model.parameters(), MAX_GRAD_NORM)

            previous_scale = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()

            if scaler.get_scale() >= previous_scale:
                scheduler.step()

            optimizer.zero_grad(set_to_none=True)

    actual, predicted = collect_predictions(cross_model, val_loader)
    val_f1 = f1_score(
        actual,
        predicted,
        labels=list(LABEL_TO_ID.values()),
        average="macro",
        zero_division=0,
    )

    print(
        f"Epoch {epoch}/{MAX_EPOCHS} | "
        f"train loss: {train_loss.item() / num_examples:.4f} | "
        f"val macro F1: {val_f1:.4f}",
        flush=True,
    )

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_epoch = epoch
        epochs_without_improvement = 0
        torch.save(cross_model.state_dict(), cross_checkpoint)
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            print("Early stopping.")
            break

print(f"Best validation macro F1: {best_val_f1:.4f} at epoch {best_epoch}")
print(f"Checkpoint: {cross_checkpoint}")

In [ ]:
# Reload the best Cross-Attention weights and evaluate on the held-out test split.
# Print metrics in the same format as the Early Fusion results.

from sklearn.metrics import accuracy_score, classification_report

assert cross_checkpoint.is_file(), f"Checkpoint missing: {cross_checkpoint}"

cross_model.load_state_dict(
    torch.load(cross_checkpoint, map_location="cpu", weights_only=True)
)

actual, predicted = collect_predictions(cross_model, loaders["test"])

label_names = sorted(LABEL_TO_ID, key=LABEL_TO_ID.get)
label_ids = [LABEL_TO_ID[name] for name in label_names]

test_accuracy = accuracy_score(actual, predicted)
test_macro_f1 = f1_score(
    actual,
    predicted,
    labels=label_ids,
    average="macro",
    zero_division=0,
)

print(f"Test accuracy: {test_accuracy:.4f}")
print(f"Test macro F1: {test_macro_f1:.4f}")
print(classification_report(
    actual,
    predicted,
    labels=label_ids,
    target_names=label_names,
    digits=4,
    zero_division=0,
))

In [ ]:
# Load the best Model 2 checkpoint and collect test probabilities in row order.

from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader

checkpoint = Path(globals().get("cross_checkpoint") or
                  "/kaggle/working/checkpoints/cross_attention_best.pt")
assert checkpoint.is_file(), f"Model 2 checkpoint missing: {checkpoint}"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model2 = CrossAttentionFusion()
model2.load_state_dict(torch.load(checkpoint, map_location="cpu", weights_only=True))
model2 = model2.to(device).eval()

test_dataset = loaders["test"].dataset
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_batch,
    num_workers=0,
)

label_ids = sorted(LABEL_TO_ID.values())
label_names = sorted(LABEL_TO_ID, key=LABEL_TO_ID.get)
true_batches, probability_batches = [], []

with torch.inference_mode():
    for batch_data in test_loader:
        true_batches.append(batch_data["labels"].numpy())
        inputs = {
            name: tensor.to(device)
            for name, tensor in batch_data.items()
            if name != "labels"
        }

        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=device.type == "cuda",
        ):
            logits = model2(**inputs)

        probability_batches.append(logits.float().softmax(dim=1).cpu().numpy())

y_true = np.concatenate(true_batches)
probabilities = np.concatenate(probability_batches)
y_pred = np.asarray(label_ids)[probabilities.argmax(axis=1)]
confidence = probabilities.max(axis=1)

print(f"Test examples: {len(y_true):,}")
print(f"Accuracy: {(y_pred == y_true).mean():.4f}")

In [ ]:
# Load the best Model 2 checkpoint and collect test probabilities in row order.

from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader

checkpoint = Path(globals().get("cross_checkpoint") or
                  "/kaggle/working/checkpoints/cross_attention_best.pt")
assert checkpoint.is_file(), f"Model 2 checkpoint missing: {checkpoint}"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model2 = CrossAttentionFusion()
model2.load_state_dict(torch.load(checkpoint, map_location="cpu", weights_only=True))
model2 = model2.to(device).eval()

test_dataset = loaders["test"].dataset
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_batch,
    num_workers=0,
)

label_ids = sorted(LABEL_TO_ID.values())
label_names = sorted(LABEL_TO_ID, key=LABEL_TO_ID.get)
true_batches, probability_batches = [], []

with torch.inference_mode():
    for batch_data in test_loader:
        true_batches.append(batch_data["labels"].numpy())
        inputs = {
            name: tensor.to(device)
            for name, tensor in batch_data.items()
            if name != "labels"
        }

        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=device.type == "cuda",
        ):
            logits = model2(**inputs)

        probability_batches.append(logits.float().softmax(dim=1).cpu().numpy())

y_true = np.concatenate(true_batches)
probabilities = np.concatenate(probability_batches)
y_pred = np.asarray(label_ids)[probabilities.argmax(axis=1)]
confidence = probabilities.max(axis=1)

print(f"Test examples: {len(y_true):,}")
print(f"Accuracy: {(y_pred == y_true).mean():.4f}")

In [ ]:
# Save confidence curves, a precision–recall plot, two confusion matrices,
# and a panel of test images annotated with Model 2 predictions.

from shutil import make_archive
from textwrap import fill

import matplotlib.pyplot as plt
from IPython.display import display
from PIL import Image
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    average_precision_score,
    confusion_matrix,
    precision_recall_curve,
)

output_dir = Path("/kaggle/working/model2_outputs")
output_dir.mkdir(parents=True, exist_ok=True)


def save_figure(fig, filename):
    fig.tight_layout()
    fig.savefig(output_dir / filename, dpi=200, bbox_inches="tight")
    display(fig)
    plt.close(fig)


# Confidence curves. A rejected prediction cannot count as a true positive.
thresholds = np.linspace(0, 1, 101)
metrics = {
    "Precision": np.zeros((len(thresholds), len(label_ids))),
    "Recall": np.zeros((len(thresholds), len(label_ids))),
    "F1": np.zeros((len(thresholds), len(label_ids))),
}

for column, label_id in enumerate(label_ids):
    is_true = y_true == label_id
    is_predicted = y_pred == label_id
    support = is_true.sum()

    for row, threshold in enumerate(thresholds):
        kept = is_predicted & (confidence >= threshold)
        tp = (kept & is_true).sum()
        predicted_count = kept.sum()

        metrics["Precision"][row, column] = tp / max(predicted_count, 1)
        metrics["Recall"][row, column] = tp / max(support, 1)
        metrics["F1"][row, column] = 2 * tp / max(predicted_count + support, 1)

for metric, values in metrics.items():
    fig, ax = plt.subplots(figsize=(7, 5))
    for column, label in enumerate(label_names):
        ax.plot(thresholds, values[:, column], label=label)
    ax.plot(thresholds, values.mean(axis=1), "k--", label="Macro average")
    ax.set(
        title=f"Model 2 | {metric} vs confidence",
        xlabel="Minimum prediction confidence",
        ylabel=metric,
        xlim=(0, 1),
        ylim=(0, 1.02),
    )
    ax.grid(alpha=0.25)
    ax.legend()
    save_figure(fig, f"{metric.lower()}_confidence.png")

# Precision–recall curves using all scores for each class.
fig, ax = plt.subplots(figsize=(7, 5))
for column, (label_id, label) in enumerate(zip(label_ids, label_names)):
    binary_true = y_true == label_id
    scores = probabilities[:, column]
    precision, recall, _ = precision_recall_curve(binary_true, scores)
    ap = average_precision_score(binary_true, scores)
    ax.step(recall, precision, where="post", label=f"{label} (AP {ap:.3f})")

ax.set(
    title="Model 2 | Precision–recall",
    xlabel="Recall",
    ylabel="Precision",
    xlim=(0, 1),
    ylim=(0, 1.02),
)
ax.grid(alpha=0.25)
ax.legend()
save_figure(fig, "precision_recall.png")

# Confusion matrices. The normalized matrix divides each row by its true-class count.
for normalization, filename, title, fmt in [
    (None, "confusion_matrix_original.png", "Counts", "d"),
    ("true", "confusion_matrix_normalized.png", "Normalized by true class", ".1%"),
]:
    matrix = confusion_matrix(
        y_true, y_pred, labels=label_ids, normalize=normalization
    )
    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay(matrix, display_labels=label_names).plot(
        ax=ax, cmap="Blues", values_format=fmt, colorbar=False
    )
    ax.set_title(f"Model 2 | Confusion matrix ({title})")
    save_figure(fig, filename)

# Show one correct and one incorrect prediction per true sentiment class.
id_to_label = dict(zip(label_ids, label_names))
fig, axes = plt.subplots(len(label_ids), 2, figsize=(10, 12))

for row, label_id in enumerate(label_ids):
    for column, correct in enumerate((True, False)):
        ax = axes[row, column]
        ax.axis("off")
        matches = np.flatnonzero((y_true == label_id) & ((y_pred == y_true) == correct))

        if not len(matches):
            ax.set_title(f"No {'correct' if correct else 'incorrect'} {id_to_label[label_id]} examples")
            continue

        index = int(matches[0])
        example = test_dataset.table.iloc[index]

        with Image.open(example["image_path"]) as image:
            ax.imshow(image.convert("RGB"))

        ax.set_title(
            f"{fill(str(example['entity']), width=42)}\n"
            f"True: {id_to_label[y_true[index]]} | "
            f"Predicted: {id_to_label[y_pred[index]]} ({confidence[index]:.1%})",
            color="darkgreen" if correct else "firebrick",
        )

fig.suptitle("Model 2 | Example test predictions")
save_figure(fig, "example_predictions.png")

archive = make_archive("/kaggle/working/model2_outputs", "zip", root_dir=output_dir)
print(f"Download from Kaggle Output: {archive}")

In [ ]:
# Check the saved Model 2 files and show download links.
from pathlib import Path
from IPython.display import FileLink, display

%cd /kaggle/working

for filename in (
    "model2_outputs.zip",
    "checkpoints/cross_attention_best.pt",
):
    path = Path(filename)
    if path.is_file():
        print(f"Found: {path.name} ({path.stat().st_size / 1024**2:.1f} MiB)")
        display(FileLink(str(path)))
    else:
        print(f"Missing: {path.name}")

In [ ]:
# model 3

In [ ]:
# Define Gated Fusion
# Encode the entity and post with DistilBERT and the image with MobileNetV3-Small.
# Learn a gate that mixes the two feature vectors before sentiment prediction.

import torch
from torch import nn
from torchvision.models import MobileNet_V3_Small_Weights, mobilenet_v3_small
from transformers import AutoModel


class GatedFusion(nn.Module):
    def __init__(self):
        super().__init__()

        self.text_encoder = AutoModel.from_pretrained(TEXT_MODEL)
        mobilenet = mobilenet_v3_small(
            weights=MobileNet_V3_Small_Weights.DEFAULT
        )
        self.image_encoder = mobilenet.features
        self.image_pool = nn.AdaptiveAvgPool2d(1)

        fusion_dim = 256
        self.text_projection = nn.Linear(
            self.text_encoder.config.hidden_size, fusion_dim
        )
        self.image_projection = nn.Linear(
            mobilenet.classifier[0].in_features, fusion_dim
        )
        self.gate = nn.Linear(2 * fusion_dim, fusion_dim)
        self.fusion_norm = nn.LayerNorm(fusion_dim)
        self.head = nn.Linear(fusion_dim, len(LABEL_TO_ID))

    def forward(self, input_ids, attention_mask, pixel_values):
        text_tokens = self.text_encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
        ).last_hidden_state

        text_features = self.text_projection(text_tokens[:, 0])
        image_features = self.image_encoder(pixel_values)
        image_features = self.image_projection(
            self.image_pool(image_features).flatten(1)
        )

        gate_input = torch.cat((text_features, image_features), dim=1)
        text_weight = torch.sigmoid(self.gate(gate_input))
        fused = text_weight * text_features + (1 - text_weight) * image_features

        return self.head(self.fusion_norm(fused))

In [ ]:
# Train and save the best checkpoint

In [ ]:
# Fine-tune both encoders and train the gate using the shared hyperparameters.
# Save the best checkpoint by validation macro F1 and apply early stopping.

from math import ceil
from pathlib import Path

from sklearn.metrics import f1_score
from transformers import get_linear_schedule_with_warmup, set_seed

assert torch.cuda.is_available(), "Enable the Kaggle GPU before training."
device = torch.device("cuda")
set_seed(SEED)

gated_model = GatedFusion().to(device)

fusion_parameters = [
    parameter
    for name, parameter in gated_model.named_parameters()
    if not name.startswith(("text_encoder.", "image_encoder."))
]

optimizer = torch.optim.AdamW(
    [
        {"params": gated_model.text_encoder.parameters(), "lr": ENCODER_LR},
        {"params": gated_model.image_encoder.parameters(), "lr": ENCODER_LR},
        {"params": fusion_parameters, "lr": FUSION_HEAD_LR},
    ],
    weight_decay=WEIGHT_DECAY,
)

train_loader = loaders["train"]
val_loader = loaders["val"]
num_batches = len(train_loader)
num_examples = len(train_loader.dataset)
total_updates = MAX_EPOCHS * ceil(num_batches / GRAD_ACCUM_STEPS)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(WARMUP_RATIO * total_updates),
    num_training_steps=total_updates,
)
scaler = torch.amp.GradScaler("cuda")
loss_fn = nn.CrossEntropyLoss(reduction="sum")

gated_checkpoint = Path("/kaggle/working/checkpoints/gated_best.pt")
gated_checkpoint.parent.mkdir(parents=True, exist_ok=True)

best_val_f1 = float("-inf")
best_epoch = 0
epochs_without_improvement = 0

for epoch in range(1, MAX_EPOCHS + 1):
    gated_model.train()
    optimizer.zero_grad(set_to_none=True)
    train_loss = torch.zeros((), device=device)

    for step, batch_data in enumerate(train_loader):
        inputs = {name: tensor.to(device) for name, tensor in batch_data.items()}
        targets = inputs.pop("labels")

        # Account for the final accumulation group if it contains fewer examples.
        group_start = (step // GRAD_ACCUM_STEPS) * GRAD_ACCUM_STEPS
        group_examples = min(
            BATCH_SIZE * GRAD_ACCUM_STEPS,
            num_examples - group_start * BATCH_SIZE,
        )

        with torch.autocast("cuda", dtype=torch.float16):
            loss = loss_fn(gated_model(**inputs), targets)

        train_loss += loss.detach()
        scaler.scale(loss / group_examples).backward()

        end_of_group = (step + 1) % GRAD_ACCUM_STEPS == 0
        if end_of_group or step + 1 == num_batches:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(gated_model.parameters(), MAX_GRAD_NORM)

            previous_scale = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()

            if scaler.get_scale() >= previous_scale:
                scheduler.step()

            optimizer.zero_grad(set_to_none=True)

    gated_model.eval()
    actual, predicted = [], []

    with torch.inference_mode():
        for batch_data in val_loader:
            actual.extend(batch_data["labels"].tolist())
            inputs = {
                name: tensor.to(device)
                for name, tensor in batch_data.items()
                if name != "labels"
            }

            with torch.autocast("cuda", dtype=torch.float16):
                logits = gated_model(**inputs)

            predicted.extend(logits.argmax(dim=1).cpu().tolist())

    val_f1 = f1_score(
        actual,
        predicted,
        labels=sorted(LABEL_TO_ID.values()),
        average="macro",
        zero_division=0,
    )

    print(
        f"Epoch {epoch}/{MAX_EPOCHS} | "
        f"train loss: {train_loss.item() / num_examples:.4f} | "
        f"val macro F1: {val_f1:.4f}",
        flush=True,
    )

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_epoch = epoch
        epochs_without_improvement = 0
        torch.save(gated_model.state_dict(), gated_checkpoint)
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            print("Early stopping.")
            break

print(f"Best validation macro F1: {best_val_f1:.4f} at epoch {best_epoch}")
print(f"Checkpoint: {gated_checkpoint}")

In [ ]:
# Evaluate the best Gated Fusion checkpoint on the held-out test split.
# Save four curves, two confusion matrices, and an annotated prediction panel.

from pathlib import Path
from shutil import make_archive
from textwrap import fill

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import display
from PIL import Image
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    precision_recall_curve,
)
from torch.utils.data import DataLoader

checkpoint = Path("/kaggle/working/checkpoints/gated_best.pt")
assert checkpoint.is_file(), f"Model 3 checkpoint missing: {checkpoint}"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model3 = GatedFusion()
model3.load_state_dict(torch.load(checkpoint, map_location="cpu", weights_only=True))
model3 = model3.to(device).eval()

# Keep predictions in dataset row order for the image examples.
test_dataset = loaders["test"].dataset
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_batch,
    num_workers=0,
)

label_names = sorted(LABEL_TO_ID, key=LABEL_TO_ID.get)
label_ids = np.array([LABEL_TO_ID[name] for name in label_names])
id_to_label = dict(zip(label_ids, label_names))
true_batches, probability_batches = [], []

with torch.inference_mode():
    for batch_data in test_loader:
        true_batches.append(batch_data["labels"].cpu().numpy())
        inputs = {
            name: tensor.to(device)
            for name, tensor in batch_data.items()
            if name != "labels"
        }

        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=device.type == "cuda",
        ):
            logits = model3(**inputs)

        probability_batches.append(logits.float().softmax(dim=1).cpu().numpy())

y_true = np.concatenate(true_batches)
probabilities = np.concatenate(probability_batches)
y_pred = label_ids[probabilities.argmax(axis=1)]
confidence = probabilities.max(axis=1)

print(f"Test examples: {len(y_true):,}")
print(f"Test accuracy: {accuracy_score(y_true, y_pred):.4f}")
print(classification_report(
    y_true, y_pred, labels=label_ids, target_names=label_names,
    digits=4, zero_division=0,
))

output_dir = Path("/kaggle/working/model3_outputs")
output_dir.mkdir(parents=True, exist_ok=True)


def save_figure(fig, filename):
    fig.tight_layout()
    fig.savefig(output_dir / filename, dpi=200, bbox_inches="tight")
    display(fig)
    plt.close(fig)


# Confidence curves. Rejected predictions still count as misses for their true class.
thresholds = np.linspace(0, 1, 101)
scores = {name: [] for name in ("Precision", "Recall", "F1")}

for label_id in label_ids:
    true_class = y_true == label_id
    kept = (y_pred == label_id) & (confidence[:, None] >= thresholds).T
    tp = (kept & true_class).sum(axis=1)
    predicted_count = kept.sum(axis=1)
    support = true_class.sum()

    scores["Precision"].append(tp / np.maximum(predicted_count, 1))
    scores["Recall"].append(tp / max(support, 1))
    scores["F1"].append(2 * tp / np.maximum(predicted_count + support, 1))

for metric, class_curves in scores.items():
    fig, ax = plt.subplots(figsize=(7, 5))
    for label, curve in zip(label_names, class_curves):
        ax.plot(thresholds, curve, label=label)
    ax.plot(thresholds, np.mean(class_curves, axis=0), "k--", label="Macro average")
    ax.set(
        title=f"Model 3 | {metric} vs confidence",
        xlabel="Minimum prediction confidence",
        ylabel=metric,
        xlim=(0, 1),
        ylim=(0, 1.02),
    )
    ax.grid(alpha=0.25)
    ax.legend()
    save_figure(fig, f"{metric.lower()}_confidence.png")

# One-vs-rest precision–recall curves using each class's probability.
fig, ax = plt.subplots(figsize=(7, 5))
for column, (label_id, label) in enumerate(zip(label_ids, label_names)):
    binary_true = y_true == label_id
    class_probabilities = probabilities[:, column]
    precision, recall, _ = precision_recall_curve(binary_true, class_probabilities)
    ap = average_precision_score(binary_true, class_probabilities)
    ax.step(recall, precision, where="post", label=f"{label} (AP {ap:.3f})")

ax.set(
    title="Model 3 | Precision–recall",
    xlabel="Recall",
    ylabel="Precision",
    xlim=(0, 1),
    ylim=(0, 1.02),
)
ax.grid(alpha=0.25)
ax.legend()
save_figure(fig, "precision_recall.png")

# Confusion matrices with raw counts and percentages within each true class.
for normalization, suffix, title, fmt in [
    (None, "original", "Counts", "d"),
    ("true", "normalized", "Normalized by true class", ".1%"),
]:
    matrix = confusion_matrix(
        y_true, y_pred, labels=label_ids, normalize=normalization
    )
    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay(matrix, display_labels=label_names).plot(
        ax=ax, cmap="Blues", values_format=fmt, colorbar=False
    )
    ax.set_title(f"Model 3 | Confusion matrix ({title})")
    save_figure(fig, f"confusion_matrix_{suffix}.png")

# Show the first correct and incorrect prediction for each true class.
fig, axes = plt.subplots(len(label_ids), 2, figsize=(10, 12))
for row, label_id in enumerate(label_ids):
    for column, correct in enumerate((True, False)):
        ax = axes[row, column]
        ax.axis("off")
        matches = np.flatnonzero(
            (y_true == label_id) & ((y_true == y_pred) == correct)
        )

        if matches.size == 0:
            outcome = "correct" if correct else "incorrect"
            ax.set_title(f"No {outcome} {id_to_label[label_id]} examples")
            continue

        index = int(matches[0])
        example = test_dataset.table.iloc[index]
        with Image.open(example["image_path"]) as image:
            ax.imshow(image.convert("RGB"))

        ax.set_title(
            f"{fill(str(example['entity']), width=42)}\n"
            f"True: {id_to_label[y_true[index]]} | "
            f"Predicted: {id_to_label[y_pred[index]]} ({confidence[index]:.1%})",
            color="darkgreen" if correct else "firebrick",
        )

fig.suptitle("Model 3 | Example test predictions")
save_figure(fig, "example_predictions.png")

archive = make_archive(str(output_dir), "zip", root_dir=output_dir)
print(f"Download from Kaggle Output: {archive}")

In [ ]:
%cd /kaggle/working

from pathlib import Path
from IPython.display import FileLink, display

archive = Path("model3_outputs.zip")

if archive.is_file():
    print(f"Found {archive.name} ({archive.stat().st_size / 1024**2:.1f} MiB)")
    display(FileLink(str(archive)))
else:
    print("The ZIP has not been created in this session.")
    print("Model 3 checkpoint exists:", Path("checkpoints/gated_best.pt").is_file())

In [ ]:
%cd /kaggle/working

from pathlib import Path
from shutil import copy2
from IPython.display import FileLink, display

source = Path("checkpoints/gated_best.pt")
download = Path("gated_best.pt")

if source.is_file():
    copy2(source, download)

if download.is_file():
    print(f"Checkpoint ready ({download.stat().st_size / 1024**2:.1f} MiB)")
    display(FileLink(download.name))
else:
    print("gated_best.pt is missing from this session.")